# Evolutionary Protein Optimization with MOSES

This notebook demonstrates evolutionary optimization of proteins using the MOSES (Meta-Optimizing Semantic Evolutionary Search) component of the bio-cognitive framework.

## 🧬 Evolutionary Design Principles

We'll explore:
- Multi-objective fitness optimization
- Pareto-optimal protein variants
- Evolutionary search strategies
- Fitness landscape exploration
- Population dynamics and convergence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EchoCog/echo-esm/blob/main/examples/evolutionary_protein_optimization.ipynb)

In [ ]:
# Install required packages
%pip install esm numpy matplotlib pandas seaborn plotly py3dmol ipywidgets scipy deap

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import py3dmol
from scipy import stats
from scipy.optimize import minimize
import random
import time
from typing import List, Dict, Tuple, Any
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('✓ Libraries imported and environment configured')

## 🔬 MOSES Evolutionary Framework

Let's implement the Meta-Optimizing Semantic Evolutionary Search system for protein optimization:

In [ ]:
# MOSES Evolutionary Framework Implementation
from enum import Enum
from dataclasses import dataclass, field
import uuid
import copy

class FitnessObjective(Enum):
    """Multi-objective fitness components"""
    STABILITY = "stability"
    SOLUBILITY = "solubility"
    BINDING_AFFINITY = "binding_affinity"
    CATALYTIC_EFFICIENCY = "catalytic_efficiency"
    IMMUNOGENICITY = "immunogenicity"
    EXPRESSION_LEVEL = "expression_level"

class MutationType(Enum):
    """Types of mutations for evolutionary search"""
    POINT_MUTATION = "point_mutation"
    INSERTION = "insertion"
    DELETION = "deletion"
    DOMAIN_SHUFFLE = "domain_shuffle"
    RECOMBINATION = "recombination"

@dataclass
class ProteinVariant:
    """Individual protein variant in evolutionary population"""
    sequence: str
    variant_id: str
    generation: int
    parent_ids: List[str]
    mutations: List[Dict[str, Any]]
    fitness_scores: Dict[str, float] = field(default_factory=dict)
    pareto_rank: int = 0
    crowding_distance: float = 0.0
    
    def __post_init__(self):
        if not self.variant_id:
            self.variant_id = f"VAR_{str(uuid.uuid4())[:8]}"

@dataclass
class EvolutionaryParameters:
    """Parameters for evolutionary optimization"""
    population_size: int = 50
    num_generations: int = 100
    mutation_rate: float = 0.1
    crossover_rate: float = 0.7
    elitism_ratio: float = 0.1
    tournament_size: int = 3
    diversity_weight: float = 0.2

class ProteinMOSES:
    """Meta-Optimizing Semantic Evolutionary Search for proteins"""
    
    def __init__(self, params: EvolutionaryParameters = None):
        self.params = params or EvolutionaryParameters()
        self.population: List[ProteinVariant] = []
        self.fitness_objectives = list(FitnessObjective)
        self.generation_history = []
        self.pareto_front_history = []
        
        # Amino acid substitution matrix (simplified)
        self.aa_similarity = self._create_aa_similarity_matrix()
        
    def _create_aa_similarity_matrix(self) -> Dict[str, List[str]]:
        """Create amino acid similarity groups for intelligent mutations"""
        return {
            'A': ['V', 'I', 'L', 'G'],  # Small/nonpolar
            'R': ['K', 'H'],            # Positive
            'N': ['Q', 'S', 'T'],       # Polar uncharged
            'D': ['E'],                 # Negative
            'C': ['M'],                 # Sulfur-containing
            'Q': ['N', 'E', 'K'],       # Polar
            'E': ['D', 'Q'],            # Negative
            'G': ['A', 'S'],            # Small
            'H': ['R', 'K', 'Y'],       # Positive/aromatic
            'I': ['L', 'V', 'M'],       # Hydrophobic branched
            'L': ['I', 'V', 'M', 'F'],  # Hydrophobic
            'K': ['R', 'H'],            # Positive
            'M': ['L', 'I', 'C'],       # Hydrophobic
            'F': ['Y', 'W', 'L'],       # Aromatic
            'P': ['G'],                 # Conformationally restricted
            'S': ['T', 'N', 'G'],       # Small polar
            'T': ['S', 'N'],            # Polar
            'W': ['F', 'Y'],            # Aromatic large
            'Y': ['F', 'W', 'H'],       # Aromatic
            'V': ['I', 'L', 'A']        # Hydrophobic branched
        }
    
    def initialize_population(self, seed_sequence: str) -> List[ProteinVariant]:
        """Initialize population with variants of seed sequence"""
        self.population = []
        
        # Add original sequence
        original = ProteinVariant(
            sequence=seed_sequence,
            variant_id="SEED_000",
            generation=0,
            parent_ids=[],
            mutations=[]
        )
        self.population.append(original)
        
        # Generate initial variants
        for i in range(1, self.params.population_size):
            variant_seq = self._mutate_sequence(seed_sequence, mutation_rate=0.05)
            variant = ProteinVariant(
                sequence=variant_seq,
                variant_id=f"INIT_{i:03d}",
                generation=0,
                parent_ids=["SEED_000"],
                mutations=[{'type': 'random_initialization', 'positions': 'multiple'}]
            )
            self.population.append(variant)
        
        # Evaluate initial fitness
        for variant in self.population:
            variant.fitness_scores = self._evaluate_fitness(variant.sequence)
        
        return self.population
    
    def _mutate_sequence(self, sequence: str, mutation_rate: float = None) -> str:
        """Apply mutations to sequence"""
        if mutation_rate is None:
            mutation_rate = self.params.mutation_rate
        
        seq_list = list(sequence)
        
        for i in range(len(seq_list)):
            if random.random() < mutation_rate:
                original_aa = seq_list[i]
                if original_aa in self.aa_similarity:
                    # Conservative mutation to similar amino acid
                    similar_aas = self.aa_similarity[original_aa]
                    seq_list[i] = random.choice(similar_aas)
                else:
                    # Random mutation
                    seq_list[i] = random.choice('ACDEFGHIKLMNPQRSTVWY')
        
        return ''.join(seq_list)
    
    def _crossover_sequences(self, parent1: str, parent2: str) -> Tuple[str, str]:
        """Perform crossover between two sequences"""
        if len(parent1) != len(parent2):
            # Handle different lengths by padding
            max_len = max(len(parent1), len(parent2))
            parent1 = parent1.ljust(max_len, 'A')
            parent2 = parent2.ljust(max_len, 'A')
        
        # Two-point crossover
        length = len(parent1)
        point1 = random.randint(1, length - 2)
        point2 = random.randint(point1 + 1, length - 1)
        
        child1 = parent1[:point1] + parent2[point1:point2] + parent1[point2:]
        child2 = parent2[:point1] + parent1[point1:point2] + parent2[point2:]
        
        return child1, child2
    
    def _evaluate_fitness(self, sequence: str) -> Dict[str, float]:
        """Evaluate multi-objective fitness for a protein sequence"""
        scores = {}
        
        # Stability (based on hydrophobic core and secondary structure propensity)
        hydrophobic_residues = sum(1 for aa in sequence if aa in 'AILMFPWV')
        stability_score = (hydrophobic_residues / len(sequence)) * 0.7 + np.random.normal(0.3, 0.1)
        scores['stability'] = max(0, min(1, stability_score))
        
        # Solubility (based on charged and polar residues)
        polar_residues = sum(1 for aa in sequence if aa in 'RNDQEHKSTY')
        solubility_score = (polar_residues / len(sequence)) * 0.8 + np.random.normal(0.2, 0.1)
        scores['solubility'] = max(0, min(1, solubility_score))
        
        # Binding affinity (based on aromatic and charged residues)
        binding_residues = sum(1 for aa in sequence if aa in 'FYWRHKDE')
        binding_score = (binding_residues / len(sequence)) * 0.6 + np.random.normal(0.4, 0.15)
        scores['binding_affinity'] = max(0, min(1, binding_score))
        
        # Catalytic efficiency (based on catalytic residues)
        catalytic_residues = sum(1 for aa in sequence if aa in 'HDSTNQC')
        catalytic_score = (catalytic_residues / len(sequence)) * 0.5 + np.random.normal(0.3, 0.12)
        scores['catalytic_efficiency'] = max(0, min(1, catalytic_score))
        
        # Immunogenicity (lower is better - based on human similarity)
        # Simplified: fewer rare amino acids = less immunogenic
        common_aas = 'AGLVISETKDNQRFYWHPMC'
        immunogenic_score = 1.0 - (sum(1 for aa in sequence if aa in common_aas) / len(sequence))
        scores['immunogenicity'] = max(0, min(1, immunogenic_score + np.random.normal(0, 0.05)))
        
        # Expression level (based on codon optimization proxy)
        expression_score = 0.6 + np.random.normal(0, 0.2)  # Simplified
        scores['expression_level'] = max(0, min(1, expression_score))
        
        return scores
    
    def _calculate_pareto_ranking(self):
        """Calculate Pareto ranking for multi-objective optimization"""
        # Convert fitness scores to arrays for easier comparison
        objectives = ['stability', 'solubility', 'binding_affinity', 'catalytic_efficiency', 'expression_level']
        # Note: immunogenicity is minimized, others are maximized
        
        n = len(self.population)
        pareto_ranks = [0] * n
        dominated_solutions = [[] for _ in range(n)]
        domination_counts = [0] * n
        
        # Calculate domination relationships
        for i in range(n):
            for j in range(n):
                if i != j:
                    dominates = True
                    strictly_dominates = False
                    
                    # Check each objective
                    for obj in objectives:
                        if obj == 'immunogenicity':
                            # Lower is better for immunogenicity
                            if self.population[i].fitness_scores[obj] > self.population[j].fitness_scores[obj]:
                                dominates = False
                                break
                            elif self.population[i].fitness_scores[obj] < self.population[j].fitness_scores[obj]:
                                strictly_dominates = True
                        else:
                            # Higher is better for other objectives
                            if self.population[i].fitness_scores[obj] < self.population[j].fitness_scores[obj]:
                                dominates = False
                                break
                            elif self.population[i].fitness_scores[obj] > self.population[j].fitness_scores[obj]:
                                strictly_dominates = True
                    
                    if dominates and strictly_dominates:
                        dominated_solutions[i].append(j)
                        domination_counts[j] += 1
        
        # Assign Pareto ranks
        current_front = [i for i in range(n) if domination_counts[i] == 0]
        rank = 1
        
        while current_front:
            for i in current_front:
                pareto_ranks[i] = rank
            
            next_front = []
            for i in current_front:
                for j in dominated_solutions[i]:
                    domination_counts[j] -= 1
                    if domination_counts[j] == 0:
                        next_front.append(j)
            
            current_front = next_front
            rank += 1
        
        # Update population with ranks
        for i, variant in enumerate(self.population):
            variant.pareto_rank = pareto_ranks[i]
    
    def evolve_generation(self) -> List[ProteinVariant]:
        """Evolve population for one generation"""
        # Calculate Pareto ranking
        self._calculate_pareto_ranking()
        
        # Selection: Tournament selection based on Pareto rank
        new_population = []
        
        # Elitism: Keep best individuals from current Pareto front
        elite_count = int(self.params.population_size * self.params.elitism_ratio)
        elite_individuals = sorted(self.population, key=lambda x: x.pareto_rank)[:elite_count]
        new_population.extend(elite_individuals)
        
        # Generate offspring
        while len(new_population) < self.params.population_size:
            # Tournament selection
            parent1 = self._tournament_selection()
            parent2 = self._tournament_selection()
            
            # Crossover
            if random.random() < self.params.crossover_rate:
                child1_seq, child2_seq = self._crossover_sequences(parent1.sequence, parent2.sequence)
            else:
                child1_seq, child2_seq = parent1.sequence, parent2.sequence
            
            # Mutation
            child1_seq = self._mutate_sequence(child1_seq)
            child2_seq = self._mutate_sequence(child2_seq)
            
            # Create new variants
            current_gen = max(v.generation for v in self.population) + 1
            
            child1 = ProteinVariant(
                sequence=child1_seq,
                variant_id=f"GEN{current_gen}_{len(new_population)}",
                generation=current_gen,
                parent_ids=[parent1.variant_id, parent2.variant_id],
                mutations=[{'type': 'crossover+mutation', 'generation': current_gen}]
            )
            
            child2 = ProteinVariant(
                sequence=child2_seq,
                variant_id=f"GEN{current_gen}_{len(new_population)+1}",
                generation=current_gen,
                parent_ids=[parent1.variant_id, parent2.variant_id],
                mutations=[{'type': 'crossover+mutation', 'generation': current_gen}]
            )
            
            # Evaluate fitness
            child1.fitness_scores = self._evaluate_fitness(child1.sequence)
            child2.fitness_scores = self._evaluate_fitness(child2.sequence)
            
            new_population.extend([child1, child2])
        
        # Trim to exact population size
        self.population = new_population[:self.params.population_size]
        
        return self.population
    
    def _tournament_selection(self) -> ProteinVariant:
        """Tournament selection based on Pareto rank"""
        tournament = random.sample(self.population, self.params.tournament_size)
        return min(tournament, key=lambda x: x.pareto_rank)
    
    def get_pareto_front(self) -> List[ProteinVariant]:
        """Get current Pareto front (rank 1 individuals)"""
        return [v for v in self.population if v.pareto_rank == 1]
    
    def run_evolution(self, seed_sequence: str, num_generations: int = None) -> Dict[str, Any]:
        """Run complete evolutionary optimization"""
        if num_generations is None:
            num_generations = self.params.num_generations
        
        # Initialize
        self.initialize_population(seed_sequence)
        
        evolution_history = []
        
        for generation in range(num_generations):
            # Evolve
            self.evolve_generation()
            
            # Record statistics
            pareto_front = self.get_pareto_front()
            
            gen_stats = {
                'generation': generation + 1,
                'pareto_front_size': len(pareto_front),
                'best_stability': max(v.fitness_scores['stability'] for v in pareto_front),
                'best_solubility': max(v.fitness_scores['solubility'] for v in pareto_front),
                'best_binding': max(v.fitness_scores['binding_affinity'] for v in pareto_front),
                'avg_fitness': np.mean([sum(v.fitness_scores.values()) for v in self.population]),
                'diversity': self._calculate_diversity()
            }
            
            evolution_history.append(gen_stats)
            
            if generation % 10 == 0:
                print(f"Generation {generation + 1}: Pareto front size = {len(pareto_front)}, "
                      f"Avg fitness = {gen_stats['avg_fitness']:.3f}")
        
        return {
            'final_population': self.population,
            'final_pareto_front': self.get_pareto_front(),
            'evolution_history': evolution_history,
            'best_variants': self._get_best_variants()
        }
    
    def _calculate_diversity(self) -> float:
        """Calculate population diversity"""
        sequences = [v.sequence for v in self.population]
        total_distance = 0
        count = 0
        
        for i in range(len(sequences)):
            for j in range(i + 1, len(sequences)):
                # Simple hamming distance
                distance = sum(c1 != c2 for c1, c2 in zip(sequences[i], sequences[j]))
                total_distance += distance
                count += 1
        
        return total_distance / count if count > 0 else 0
    
    def _get_best_variants(self) -> Dict[str, ProteinVariant]:
        """Get best variants for each objective"""
        best_variants = {}
        
        objectives = ['stability', 'solubility', 'binding_affinity', 'catalytic_efficiency', 'expression_level']
        
        for obj in objectives:
            best_variants[obj] = max(self.population, key=lambda x: x.fitness_scores[obj])
        
        # Best for immunogenicity (minimum)
        best_variants['immunogenicity'] = min(self.population, key=lambda x: x.fitness_scores['immunogenicity'])
        
        return best_variants

print('✓ MOSES Evolutionary Framework implemented')